# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [2]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. "
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. "
        "A report valued the project at $3.2 billion.")

print(text)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q1

In [ ]:
# import nltk
# nltk.download('punkt') # token sentences
# nltk.download('punkt_tab') 


In [ ]:
# Q1 (1 pt): Sentence splitting (regex + NLTK)
# - Use regex to protect acronyms like U.P.C. so they don't break sentence boundaries.
# - Then use nltk.sent_tokenize.
#
# Return: sentences (list of strings)
import re
import nltk
# TODO: implement protect_acronym_dots and restore_acronym_dots (or equivalent)
def protect_acronym(text):
    pattern = r"\b([A-Za-z])\."
    temporal_mark = r"\1<ACRO_DOT>" 
    protected = re.sub(pattern, temporal_mark, text)
    return protected

def restore_acronym(text):
    restored = text.replace("<ACRO_DOT", ".")
    return restored

# TODO: apply sent_tokenize
protect = protect_acronym(text)
tokenized = nltk.sent_tokenize(protect)
sentences = []
for s in tokenized:
    restored_sentence = restore_acronym(s)
    sentences.append(restored_sentence)

print(sentences)


['In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.', 'He is 1.86m tall and met with researchers from U.>P.>C.> and U.>N.>E.>S.>C.>O.> A report valued the project at $3.2 billion.']


## Q2

In [ ]:
# Q2 (1 pt): Regex normalization
# Convert:
#  - U.P.C. -> UPC, U.N.E.S.C.O. -> UNESCO (general rule: remove dots in acronyms)
#  - 1.86m -> 186 centimeters (general: X.YZm -> int(round(float(X.YZ)*100)) centimeters)
#  - $3.2 billion -> three point two billion  (digits 0-9 are enough)
#
# Return: text_norm
import re
text_norm = text
# 1) Remove dots in acronyms (U.N.E.S.C.O. -> UNESCO)
acronym_pattern = r'\b(?:[A-Za-z]\.){2,}'
def remove_acronym(match):
    acronym = match.group(0)
    no_dots = acronym.replace(".", "")
    return no_dots

text_norm = re.sub(acronym_pattern, remove_acronym, text_norm)

# 2) Meters to centimeters
meters_pattern = r'\b(\d+\.\d+)m\b'
def mtocm(match):
    number_str = match.group(1)
    number_float = float(number_str)
    centimeters = int(round(number_float * 100))
    return f"{centimeters} cm"

text_norm = re.sub(meters_pattern, mtocm, text_norm)

# 3) $3.2 billion -> three point two billion

money_pattern = r'\$(\d+\.\d+)\s+billion'
def from_money_to_words(match):
    number_str = match.group(1)
    number_letters = {
        '0': 'zero',
        '1': 'one',
        '2': 'two',
        '3': 'three',
        '4': 'four',
        '5': 'five',
        '6': 'six',
        '7': 'seven',
        '8': 'eight',
        '9': 'nine',
        '.': 'point'
    }
    word_list = []
    for ch in number_str:
        word = number_letters[ch]
        word_list.append(word)
    word = " ".join(word_list)
    result = word + " billion"
    return result

text_norm = re.sub(money_pattern, from_money_to_words, text_norm)
print(text_norm)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 cm tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.


## Q3

In [6]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
# Requirements:
# - Convert to lowercase except:
#   - Acronyms (ALL CAPS) stay uppercase (e.g., UNESCO, UPC, CEO)
#   - MixedCase tokens stay as-is (e.g., OpenAI)
#   - Multiword proper nouns joined with underscore (Sam Altman -> Sam_Altman) and preserved
#
# Return: text_case

text_case = text_norm
# 1. Multiword
pattern_multiword = r"\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b"
def join_multiword(match):
    group = match.group(1)
    parts = group.split()
    joined = " ".join(parts)
    return joined

text_case = re.sub(pattern_multiword, join_multiword, text_case)

# 2. Tokens(acronyms, mixedcase, multiword)
tokens = text_case.split()
final_tokens = []

for tkn in tokens:
    # Acronyms
    if tkn.isupper():
        final_tokens.append(tkn)
        continue
    # MixedCase
    if re.match(r'^[A-Z][a-z]+', tkn):
        final_tokens.append(tkn)
        continue
    # The rest
    lowered = tkn.lower()
    final_tokens.append(lowered)

text_case = " ".join(final_tokens)
print(text_case)

In mid-february 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 cm tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.


## Q4

In [14]:
# Q4 (1 pt): Tokenization
# Use a tokenizer of your choice (e.g., nltk.word_tokenize).
# Return: tokens (list)
import nltk
tokens = nltk.word_tokenize(text_case)
print(tokens)


['In', 'mid-february', '2026', ',', 'the', 'CEO', 'of', 'OpenAI', ',', 'Sam', 'Altman', ',', 'visited', 'Barcelona', '.', 'He', 'is', '186', 'cm', 'tall', 'and', 'met', 'with', 'researchers', 'from', 'UPC', 'and', 'UNESCO', 'A', 'report', 'valued', 'the', 'project', 'at', 'three', 'point', 'two', 'billion', '.']


## Q5

In [15]:
# Q5 (1 pt): Stopword removal
# - Remove English stopwords
# - Do NOT remove entity tokens like OpenAI, Sam_Altman, Barcelona, UNESCO, UPC
# Return: tokens_nostop
stop_words = {
    "the", "a", "an", "in", "on", "at", "of", "for", "to", "and", "or", "is", "are",
    "was", "were", "be", "been", "with", "by", "from", "that", "this", "it", "as"
}

tokens_nostop = []
for tkn in tokens:
    # If it is an entity -> no elimination
    if tkn[0].isupper() or tkn.isupper() or "_" in tkn:
        tokens_nostop.append(tkn)
        continue
    
    # If it isn't a stopword -> no elimination
    if tkn.lower() not in stop_words:
        tokens_nostop.append(tkn)

print(tokens_nostop)


['In', 'mid-february', '2026', ',', 'CEO', 'OpenAI', ',', 'Sam', 'Altman', ',', 'visited', 'Barcelona', '.', 'He', '186', 'cm', 'tall', 'met', 'researchers', 'UPC', 'UNESCO', 'A', 'report', 'valued', 'project', 'three', 'point', 'two', 'billion', '.']


## Q6

In [19]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Return: bigrams = [(w1, w2), ...]

bigrams = []
lenght = len(tokens) - 1
for i in range(lenght):
    w1 = tokens[i]
    w2 = tokens[i + 1]
    pairs = (w1, w2)
    bigrams.append(pairs)
print(bigrams)


[('In', 'mid-february'), ('mid-february', '2026'), ('2026', ','), (',', 'the'), ('the', 'CEO'), ('CEO', 'of'), ('of', 'OpenAI'), ('OpenAI', ','), (',', 'Sam'), ('Sam', 'Altman'), ('Altman', ','), (',', 'visited'), ('visited', 'Barcelona'), ('Barcelona', '.'), ('.', 'He'), ('He', 'is'), ('is', '186'), ('186', 'cm'), ('cm', 'tall'), ('tall', 'and'), ('and', 'met'), ('met', 'with'), ('with', 'researchers'), ('researchers', 'from'), ('from', 'UPC'), ('UPC', 'and'), ('and', 'UNESCO'), ('UNESCO', 'A'), ('A', 'report'), ('report', 'valued'), ('valued', 'the'), ('the', 'project'), ('project', 'at'), ('at', 'three'), ('three', 'point'), ('point', 'two'), ('two', 'billion'), ('billion', '.')]


## Q7

In [10]:
# Q7 (2 pt): Bigram Language Model + next-word prediction
# Build:
# - bigram_counts[(w1,w2)]
# - context_counts[w1]
# - model[w1][w2] = P(w2|w1) = count(w1,w2)/count(w1)
#
# Then implement:
# def predict_next(prev_word, model, top_k=3): -> list[(next_word, prob)] sorted

bigram_counts = None
context_counts = None
model = None

def predict_next(prev_word, model, top_k=3):
    # TODO
    return None

# Example:
# print(predict_next("OpenAI", model, top_k=3))


## Q8

In [11]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
#
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

merges = None

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

# print(merges)


## Q9

In [12]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = None
FP = None
FN = None
TN = None

accuracy = None
precision = None
recall = None
f1 = None

# print(accuracy, precision, recall, f1)
